In [ ]:
import json
from pathlib import Path

import pandas as pd


DATA_PATH = Path("../dadosDesafio/dados_nivel_1.json")

with DATA_PATH.open(encoding="utf-8") as file:
    dataset = json.load(file)

operacoes = pd.DataFrame(dataset["operacoes"])

taxa_usd_brl = dataset["taxa_cambio_usd_brl"]

print(f"Taxa USD/BRL: {taxa_usd_brl}")
print(f"Quantidade de operações: {len(operacoes)}")


operacoes.head()

operacoes.info()

operacoes.describe(include="all")

print("Valores ausentes:")
display(operacoes.isna().sum())

duplicados = operacoes[operacoes.duplicated(subset="id", keep=False)].sort_values("id")

display(duplicados)

operacoes = operacoes.drop_duplicates(subset="id", keep="first").copy()

operacoes["data"] = pd.to_datetime(
    operacoes["data"],
    errors="coerce"
)

print(f"Operações após deduplicação: {len(operacoes)}"),
print(f"IDs únicos: {operacoes['id'].nunique()}")
print(f"Datas ausentes: {operacoes['data'].isna().sum()}")


operacoes["valor"] = pd.to_numeric(
    operacoes["valor"],
    errors="coerce"
)


operacoes["valor_brl"] = operacoes["valor"].astype(float)


operacoes.loc[
    operacoes["moeda"] == "USD",
    "valor_brl"
] = (
    operacoes.loc[
        operacoes["moeda"] == "USD",
        "valor"
    ] * taxa_usd_brl
)

display(
    operacoes[
        ["id", "valor", "moeda", "valor_brl"]
    ]
)

volume_por_cliente = (
    operacoes.groupby("cliente_id", as_index=False)
      .agg(
          volume_total_brl=("valor_brl", "sum")
      )
      .sort_values(
          "volume_total_brl",
          ascending=False
      )
)

display(volume_por_cliente)

operacoes_por_canal = (
    operacoes.groupby("canal")
      .size()
      .reset_index(name="quantidade_operacoes")
      .sort_values(
          "quantidade_operacoes",
          ascending=False
      )
)

display(operacoes_por_canal)

operacoes_com_data = operacoes.dropna(subset=["data"]).copy()

resumo_diario = (
    operacoes_com_data
    .groupby(["cliente_id", "data"])
    .agg(
        quantidade_operacoes=("id", "count"),
        soma_brl=("valor_brl", "sum"),
        maior_operacao_brl=("valor_brl", "max")
    )
    .reset_index()
)
display(resumo_diario)

resumo_diario["regra_fracionamento"] = (
    (resumo_diario["quantidade_operacoes"] >= 3)
    &
    (resumo_diario["soma_brl"] > 50_000)
    &
    (resumo_diario["maior_operacao_brl"] < 20_000)
)

casos_fracionamento = resumo_diario[
    resumo_diario["regra_fracionamento"]
]

display(casos_fracionamento)

validacao_regra_1 = resumo_diario[
    resumo_diario["cliente_id"].isin(
        ["CLI-A-1", "CLI-A-3"]
    )
].copy()

display(validacao_regra_1)

estatisticas_cliente = (
    operacoes.groupby("cliente_id")
      .agg(
          quantidade_operacoes=("id", "count"),
          mediana_brl=("valor_brl", "median")
      )
      .reset_index()
)
display(estatisticas_cliente)

operacoes = operacoes.merge(
    estatisticas_cliente,
    on="cliente_id",
    how="left"
)

operacoes["regra_valor_atipico"] = (
    (operacoes["quantidade_operacoes"] >= 4)
    &
    (operacoes["valor_brl"] > 5 * operacoes["mediana_brl"])
)

casos_atipicos = operacoes[
    operacoes["regra_valor_atipico"]
][
    [
        "id",
        "cliente_id",
        "valor_brl",
        "quantidade_operacoes",
        "mediana_brl"
    ]
]

display(casos_atipicos)

sinais_por_cliente = (
    operacoes.groupby("cliente_id")
      .agg(
          regra_valor_atipico=("regra_valor_atipico", "any")
      )
      .reset_index()
)

display(casos_atipicos)

fracionamento_por_cliente = (
    resumo_diario.groupby("cliente_id")
    .agg(
        regra_fracionamento=("regra_fracionamento", "any")
    )
    .reset_index()
)

sinais_por_cliente = sinais_por_cliente.merge(
    fracionamento_por_cliente,
    on="cliente_id",
    how="outer"
).fillna(False)

CLIENTE_ANALISE = "CLI-A-1"

def montar_contexto_cliente(
    df: pd.DataFrame,
    sinais: pd.DataFrame,
    cliente_id: str
) -> dict:

    operacoes = df[
        df["cliente_id"] == cliente_id
    ][
        [
            "id",
            "data",
            "valor_brl",
            "canal",
            "tipo",
            "contraparte",
            "observacao"
        ]
    ].copy()

    sinais_cliente = sinais[
        sinais["cliente_id"] == cliente_id
    ].iloc[0]

    return {
        "cliente_id": cliente_id,
        "operacoes": operacoes.to_dict(
            orient="records"
        ),
        "sinais_deterministicos": {
            "fracionamento": bool(
                sinais_cliente["regra_fracionamento"]
            ),
            "valor_atipico": bool(
                sinais_cliente["regra_valor_atipico"]
            )
        }
    }

contexto = montar_contexto_cliente(
    operacoes,
    sinais_por_cliente,
    CLIENTE_ANALISE
)    

Taxa USD/BRL: 5.4
Quantidade de operações: 20
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 1.5 KB
Valores ausentes:


id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


Operações após deduplicação: 19
IDs únicos: 19
Datas ausentes: 1


,id,valor,moeda,valor_brl
0,OP-0001,18100,BRL,18100.0
1,OP-0002,17300,BRL,17300.0
2,OP-0003,18800,BRL,18800.0
3,OP-0004,3300,BRL,3300.0
4,OP-0005,25900,BRL,25900.0
5,OP-0006,27000,BRL,27000.0
6,OP-0007,17200,BRL,17200.0
7,OP-0008,15200,BRL,15200.0
8,OP-0009,16100,BRL,16100.0
10,OP-0010,3800,BRL,3800.0


...

,cliente_id,data,quantidade_operacoes,soma_brl,maior_operacao_brl,regra_fracionamento
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True


,cliente_id,data,quantidade_operacoes,soma_brl,maior_operacao_brl,regra_fracionamento
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True
1,CLI-A-1,2026-03-21,1,3300.0,3300.0,False
3,CLI-A-3,2026-03-05,3,48500.0,17200.0,False


,cliente_id,quantidade_operacoes,mediana_brl
0,CLI-A-1,4,17700.0
1,CLI-A-2,2,26450.0
2,CLI-A-3,3,16100.0
3,CLI-A-4,4,5450.0
4,CLI-A-5,4,3600.0
5,CLI-A-6,2,5100.0


,id,cliente_id,valor_brl,quantidade_operacoes,mediana_brl
12,OP-0013,CLI-A-4,64800.0,4,5450.0


,id,cliente_id,valor_brl,quantidade_operacoes,mediana_brl
12,OP-0013,CLI-A-4,64800.0,4,5450.0


In [12]:
from typing import Literal
from pydantic import BaseModel


class Parecer(BaseModel):
    nivel_risco: Literal[
        "baixo",
        "médio",
        "alto"
    ]
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str

In [13]:

cliente_escolhido = "CLI-A-4"

dados_cliente = operacoes[
    operacoes["cliente_id"] == cliente_escolhido
].copy()

display(
    dados_cliente[
        [
            "id",
            "data",
            "valor_brl",
            "moeda",
            "canal",
            "tipo",
            "contraparte",
            "observacao",
            "regra_valor_atipico"
        ]
    ]
)

,id,data,valor_brl,moeda,canal,tipo,contraparte,observacao,regra_valor_atipico
9,OP-0010,2026-03-03,3800.0,BRL,cartao,pagamento,Alfa Comercio LTDA,,False
10,OP-0011,2026-03-11,5100.0,BRL,boleto,pagamento,Beta Servicos ME,,False
11,OP-0012,2026-03-18,5800.0,BRL,pix,transferencia_enviada,Gama Distribuidora,,False
12,OP-0013,2026-03-24,64800.0,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional,True


In [14]:
prompt_1 = f"""
Você é um analista de prevenção à lavagem de dinheiro.

Analise o cliente {cliente_escolhido} utilizando somente
as informações fornecidas abaixo.

Os cálculos determinísticos já foram realizados pelo sistema.
Não realize novos cálculos.

Dados calculados:
- quantidade de operações: {len(dados_cliente)}
- mediana dos valores em BRL: {dados_cliente["valor_brl"].median():.2f}
- regra de valor atípico acionada:
  {dados_cliente["regra_valor_atipico"].any()}

Operações do cliente:

{dados_cliente[
    [
        "id",
        "data",
        "valor_brl",
        "moeda",
        "canal",
        "tipo",
        "contraparte",
        "observacao"
    ]
].to_string(index=False)}

Produza um parecer contendo:

- nivel_risco: baixo, médio ou alto
- tipologia_suspeita
- red_flags: lista de sinais observados
- justificativa

Diferencie claramente fatos observados de interpretações.

Não invente informações que não estejam nos dados.
"""

print(prompt_1)


Você é um analista de prevenção à lavagem de dinheiro.

Analise o cliente CLI-A-4 utilizando somente
as informações fornecidas abaixo.

Os cálculos determinísticos já foram realizados pelo sistema.
Não realize novos cálculos.

Dados calculados:
- quantidade de operações: 4
- mediana dos valores em BRL: 5450.00
- regra de valor atípico acionada:
  True

Operações do cliente:

     id       data  valor_brl moeda  canal                   tipo        contraparte            observacao
OP-0010 2026-03-03     3800.0   BRL cartao              pagamento Alfa Comercio LTDA                      
OP-0011 2026-03-11     5100.0   BRL boleto              pagamento   Beta Servicos ME                      
OP-0012 2026-03-18     5800.0   BRL    pix  transferencia_enviada Gama Distribuidora                      
OP-0013 2026-03-24    64800.0   USD    ted transferencia_recebida    Zeta Importacao remessa internacional

Produza um parecer contendo:

- nivel_risco: baixo, médio ou alto
- tipologia_suspeit

In [ ]:
from ollama import chat

resposta = chat(
   model="qwen3:8b",
    messages=[
        {
            "role": "user",
            "content": "Responda apenas: Ollama funcionando."
        }
    ]
)

print(resposta.message.content)

ResponseError: model 'llama3.1:8b' not found (status code: 404)

In [ ]:
resposta = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": "Você é um analista de prevenção à lavagem de dinheiro."
        },
        {
            "role": "user",
            "content": prompt_1
        }
    ],
    text_format=Parecer
)

parecer = resposta.output_parsed

print(parecer)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
print("Nível de risco:", parecer.nivel_risco)
print("Tipologia:", parecer.tipologia_suspeita)

print("\nRed flags:")
for flag in parecer.red_flags:
    print("-", flag)

print("\nJustificativa:")
print(parecer.justificativa)

# Desafio Itaú Estágio Engenharia de IA — Nível 1
## Prevenção à Lavagem de Dinheiro

Este notebook apresenta o tratamento dos dados, a aplicação de regras
determinísticas e a análise interpretativa de um cliente sinalizado
utilizando um modelo de linguagem.

A solução mantém separadas as responsabilidades:

- pandas: limpeza, agregações e cálculos determinísticos;
- regras: identificação objetiva dos sinais definidos no desafio;
- LLM: interpretação dos sinais e redação do parecer;
- validação estruturada: garantia de que a resposta da LLM segue o formato esperado.

### Validação da Regra 1

O cliente CLI-A-1 é corretamente sinalizado: realizou 3 operações
no mesmo dia, totalizando R$ 54.200, sem nenhuma operação individual
igual ou superior a R$ 20.000.

O cliente CLI-A-3 apresenta comportamento semelhante, porém o total
das três operações é R$ 48.500. Como esse valor não ultrapassa
R$ 50.000, o cliente não é sinalizado.

A comparação demonstra que a regra considera simultaneamente os três
critérios definidos, e não apenas a quantidade de operações.

## Tratamento dos problemas de qualidade

Foram identificados dois problemas relevantes:

1. O identificador `OP-0007` aparece duas vezes com os mesmos valores
   em todos os campos. A segunda ocorrência foi considerada uma duplicação
   do registro e apenas uma ocorrência foi mantida.

2. A operação `OP-0017` possui data ausente. A data não foi inferida ou
   preenchida artificialmente, pois não existe evidência suficiente para
   determinar o dia correto. Essa operação permanece na base para as
   análises que não dependem de data, mas não participa da regra de
   fracionamento, que exige operações realizadas na mesma data.

A deduplicação é importante porque uma duplicação alteraria contagens,
medianas e agregações. Já a preservação da operação sem data evita a
introdução de uma informação que não está presente na fonte.